# Data Science for Business - Convolutional Neural Network (CNN) on Fashion MNIST

*Session 9 · ISLP 10.3.1–10.3.4 (convolution, pooling, architecture, data augmentation)*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
# Install packages that are not pre-installed on Colab
# !pip install torchinfo torchmetrics

In [ ]:
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets
from torchvision.transforms import v2
from torchinfo import summary
from torchmetrics import Accuracy

In [ ]:
torch.manual_seed(42)

Check if a GPU is available: an NVIDIA GPU (`cuda`, e.g., on Colab), an Apple Silicon GPU (`mps`), or otherwise the CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
device

## Problem description

The Fashion MNIST dataset is a dataset of thumbnails of fashion articles from Zalando (28x28 grayscale images). The dataset consists of a training set of 60,000 examples and a test set of 10,000 examples. Each image is associated with a label from 10 classes (e.g., Shirts, Sneakers, Trousers...). The dataset is balanced in terms of label distribution.

You can have a look at the dataset here: https://github.com/zalandoresearch/fashion-mnist

Your task is to train a CNN classifier to predict the class of the fashion articles. Fashion MNIST is a drop-in replacement for the MNIST handwritten digits of ISLP 10.2 (Figure 10.3), but it is harder: a simple MLP reaches about 97% test accuracy on the digits, so don't expect that here.

ISLP's Python code uses PyTorch together with `pytorch_lightning`, which hides the training loop. Here we write the loop out by hand; the steps are the same, only the syntax differs.

## Load data

The dataset is built into the `torchvision` library, so we can download it directly.

We can perform a number of transformations during loading. For now, we only convert the images to PyTorch tensors and scale the pixel values from integers 0–255 to floats between 0 and 1.

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

In [ ]:
train_dataset = datasets.FashionMNIST(root='data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='data', train=False, transform=transform, download=True)

Let's check the dimensions of the training and test sets.

In [ ]:
print(train_dataset.data.shape)
print(test_dataset.data.shape)

Take a look at the first image in the training set. The raw data are stored as integers between 0 (black) and 255 (white).

In [ ]:
train_dataset.data[0]

We can take this tensor of integers and pass it to the `imshow()` function of `matplotlib` to visualize the image. Looks like a boot!

In [ ]:
plt.imshow(train_dataset.data[0], cmap='gray')
plt.axis('off')
plt.show()

What is the label of this image?

In [ ]:
print("Label:", train_dataset.targets[0].item(), "=", train_dataset.classes[train_dataset.targets[0]])
print("List of labels:", train_dataset.classes)

When we index the dataset (instead of its raw `.data`), the transformation is applied: we get a float tensor of shape (channels, height, width) = (1, 28, 28) with values between 0 and 1, plus the label.

In [ ]:
image, label = train_dataset[0]
print(image.shape, image.dtype, image.min().item(), image.max().item(), label)

## Prepare data

### Validation set

To monitor overfitting during training, we hold out 20% of the training images as a validation set (ISLP 5.1.1). The test set stays untouched until the very end.

In [ ]:
train_subset, val_subset = random_split(train_dataset, [48000, 12000], generator=torch.Generator().manual_seed(42))
print(len(train_subset), len(val_subset), len(test_dataset))

### Data loaders

A `DataLoader` serves minibatches of 64 images; each optimizer step uses one minibatch's gradient (stochastic gradient descent, ISLP 10.7.2). `shuffle=True` reshuffles the training data in every epoch.

In [ ]:
batch_size = 64

train_loader = DataLoader(dataset=train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

## Network architecture

Like with MLPs, we define the architecture of a CNN by creating a class that inherits from `torch.nn.Module`.

Our simple CNN has the following architecture (ISLP 10.3):
- Convolution layer with 16 convolution filters of size 3x3 (kernel), a stride of 1 and padding of 1. Each filter produces one feature map, so the output has 16 channels (ISLP 10.3.1).
- ReLU activation function (the *detector layer* in ISLP 10.3.1).
- Max pooling layer with a kernel size of 2x2 and stride of 2 (ISLP 10.3.2).
- A layer without parameters that flattens the feature maps into a vector.
- Fully connected layer with 128 hidden units.
- ReLU activation function.
- Fully connected output layer with 10 units (one for each class).

In [ ]:
# Define CNN model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(16 * 14 * 14, 128) # 16 feature maps of size 14x14 after pooling
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

Where does `16 * 14 * 14` come from? For an input of width $W$, kernel size $K$, padding $P$ and stride $S$, a convolution layer outputs feature maps of width $(W + 2P - K)/S + 1$. Here $(28 + 2 - 3)/1 + 1 = 28$: with a 3x3 filter, `padding=1` keeps the image size unchanged (ISLP 10.3.1; equivalent to `padding='same'`). The 2x2 max pooling then halves each side to 14 (ISLP 10.3.2). So 16 feature maps (channels) × 14 × 14 = 3,136 numbers are flattened and fed into the fully connected layer (ISLP 10.3.3). Note that the 16 filters have only $16 \cdot (3 \cdot 3 + 1) = 160$ parameters, whereas the fully connected layer has $3{,}136 \cdot 128 + 128 = 401{,}536$.

Let's initialize the CNN class and specify an optimizer and loss function. `CrossEntropyLoss` is the negative multinomial log-likelihood, also known as cross-entropy (ISLP eq. 10.14), the common loss function for multi-class classification problems. Adam is a variant of stochastic gradient descent (ISLP 10.7.2); `lr` is the learning rate ρ (ISLP eq. 10.27).

In [ ]:
learning_rate = 0.001
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

`torchinfo.summary` shows the output shape and number of parameters of each layer for a minibatch of 64 images (ISLP uses the same package). We pass `device` so that the model stays on the GPU.

In [ ]:
summary(model, input_size=(batch_size, 1, 28, 28), device=device)

## Train the network

We iterate over the training set in minibatches, make predictions, calculate the loss, and update the weights of the network (backpropagation, ISLP 10.7.1). One epoch = one pass over the 48,000 training images (750 minibatches of 64). After each epoch we compute the loss on the validation set without updating the weights (`model.eval()` and `torch.no_grad()`). The training loop is the same as for the MLP.

Networks like this are often trained for 30 epochs or more (e.g., with minibatches of 256); to save time in class, we train for 5 epochs.

In [ ]:
num_epochs = 5

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Validation loss (no weight updates)
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_loss += criterion(model(images), labels).item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Training loss: {running_loss/len(train_loader):.4f}, Validation loss: {val_loss/len(val_loader):.4f}")

The training loss keeps falling with every epoch; only the held-out validation loss tells us whether the network starts to overfit. If the validation loss starts rising while the training loss keeps falling, we should stop training earlier (early stopping, ISLP 10.7.2).

## Make predictions

Finally, we make predictions on the test set and calculate the accuracy of the model. Don't forget to set the model to evaluation mode before making predictions. The predicted class is the one with the largest output (`argmax`).

In [ ]:
model.eval()
accuracy_metric = Accuracy(task="multiclass", num_classes=10).to(device)

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        accuracy_metric.update(outputs.argmax(dim=1), labels)

test_accuracy = accuracy_metric.compute()
print(f"Test Accuracy: {test_accuracy.item():.3f}")

Compare this with the test accuracy of the MLP. The convolution filters look at local neighbourhoods of pixels and use the same weights at every position of the image (ISLP 10.3.1), which suits images well.

We can also compute predictions for an individual image. Let's take the first image of the training set (the ankle boot from above) and see what the model predicts. Indexing the dataset applies the same transformation as during training; `unsqueeze(0)` adds the batch dimension, i.e., a "minibatch" of one image.

In [ ]:
model.eval()
image, label = train_dataset[0]   # already scaled to [0, 1] by the transform
with torch.no_grad():
    probs = torch.softmax(model(image.unsqueeze(0).to(device)), dim=1)

pred = probs.argmax(dim=1).item()
print(probs.cpu().numpy().round(3))
print(f"Predicted label: {train_dataset.classes[pred]}, true label: {train_dataset.classes[label]}")

## Your turn!

**Your turn!** Add a second convolution–pooling block (e.g., 32 filters, as in ISLP Fig. 10.8) or a dropout layer (`nn.Dropout`, ISLP 10.7.3) to the CNN. Remember to adjust the input size of the first fully connected layer. Does the test accuracy improve?

In [ ]:
# YOUR CODE HERE